# fatAnalyze - interactive ROI quickstart

30-second demo of the user-drawn ROI feature. Runs end-to-end on a
synthetic CT so you don't need DICOM data or TotalSegmentator to try it.

Steps:
1. Set an interactive matplotlib backend (needed for `PolygonSelector`)
2. Build a small synthetic CT with a bright 80-HU cube in the middle
3. Draw a polygon around the cube (or anything else you like)
4. Run the standard histogram + clinical-metric pipeline on the ROI
5. Render a two-panel overlay + histogram figure

When you're ready to use this on a real DICOM series, jump to
`notebooks/single_case.ipynb` cell block 7 (User-drawn ROIs).

In [ ]:
# Use an interactive backend so the polygon drawer receives mouse events.
# In a fresh kernel this must run BEFORE any `import matplotlib.pyplot`.
%matplotlib widget

import numpy as np
import SimpleITK as sitk
import matplotlib.pyplot as plt

from fatanalyze.interactive import (
    draw_roi_2d, analyze_user_roi, plot_user_roi,
)

In [ ]:
# Build a 256x256x20 synthetic CT, 50 HU background, with a 80-HU cube in the middle.
arr = np.full((20, 256, 256), 50, dtype=np.int16)
arr[10, 100:180, 100:180] = 80   # 80x80 px bright region on z=10
image = sitk.GetImageFromArray(arr)
image.SetSpacing((1.0, 1.0, 5.0))   # 1 mm in-plane, 5 mm through-plane
print("image:", image.GetSize(), "spacing:", image.GetSpacing())

In [ ]:
# Open the polygon drawer on z=10 (the cube's slice). Click to add
# vertices, drag to move, then press Esc or click 'Finish'.
roi = draw_roi_2d(
    image,
    z_index=10,
    name="my_first_roi",
    preset="liver",
    title="Draw a polygon around the bright cube, then Finish",
)
if roi.empty_warning:
    print("[skipped] closed without drawing a polygon (need >= 3 vertices)")
else:
    print(f"drew {roi.n_points}-point polygon: "
          f"{roi.n_voxels} voxels, {roi.area_cm2:.1f} cm^2")

In [ ]:
# Standard histogram + clinical-metric pipeline, same code path as TS ROIs.
if not roi.empty_warning:
    result = analyze_user_roi(image, roi)
    print(f"target:        {result['target']}")
    print(f"n_voxels:      {result['n_voxels']}")
    print(f"area_cm2:      {result['area_cm2']:.1f}")
    print(f"volume_ml:     {result['volume_ml']:.1f}")
    print(f"mean_hu:       {result['mean_hu']:.1f}")
    print(f"median_hu:     {result['median_hu']:.1f}")
    print(f"std_hu:        {result['std_hu']:.1f}")
    print(f"ratios:        {result['ratios']}")
    if result['clinical_flags']:
        print(f"flags:         {result['clinical_flags']}")
    if result['psoas_metrics']:
        print(f"psoas_metrics: {result['psoas_metrics']}")
else:
    print("[skipped] no ROI drawn")

In [ ]:
# Two-panel figure: CT + polygon overlay | HU histogram with clinical bands.
if not roi.empty_warning:
    fig = plot_user_roi(image, roi, analysis=analyze_user_roi(image, roi))
    plt.show()
else:
    print("[skipped] no ROI drawn")

## Next steps

- **Run on a real DICOM series:** open `notebooks/single_case.ipynb`,
  run cells 1-6 to load + segment, then run cell block 7 (User-drawn ROIs)
  on the L3-approx slice TS found automatically.
- **Try other presets:** swap `preset="liver"` for `iliopsoas_left`,
  `iliopsoas_right`, `pancreas`, or `spleen` to use that target's HU
  ranges and clinical thresholds. Psoas presets also populate
  `psoas_metrics` (IMAT / low-density muscle / normal muscle / myosteatosis flag).
- **Save the mask for later:** `roi.save_nifti("my_roi.nii.gz")`
  writes a NIfTI mask plus a JSON sidecar with name/preset/z_index.
  Re-load with `from fatanalyze.interactive import load_user_roi; load_user_roi(path, image)`.
- **Build ROIs from code (no mouse):** if you already know the vertices,
  use `rasterize_polygon(image, z_index, vertices_xy)` and wrap the
  result in a `UserROI(...)` directly. See `tests/test_user_roi.py` for examples.